# Chess Engine — Training Analysis
Generates all figures and statistics for the report from `runs/train_log.csv` and `runs/elo_log.csv`.

In [ ]:
# Install if needed
%pip install -q matplotlib pandas seaborn

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
from pathlib import Path

ROOT    = Path('..') 
FIGDIR  = Path('figures')
FIGDIR.mkdir(exist_ok=True)

# ── style ──────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'font.family':      'serif',
    'font.size':        10,
    'axes.titlesize':   10,
    'axes.labelsize':   10,
    'legend.fontsize':  9,
    'figure.dpi':       150,
    'axes.grid':        True,
    'grid.alpha':       0.3,
    'axes.spines.top':  False,
    'axes.spines.right':False,
})

# ── load data ──────────────────────────────────────────────────────────────
train = pd.read_csv(ROOT / 'runs/train_log.csv')
elo   = pd.read_csv(ROOT / 'runs/elo_log.csv')

print(f'Train log : {len(train)} iterations  ({train.iteration.min()}–{train.iteration.max()})')
print(f'Elo log   : {len(elo)} eval points')

## Key Statistics

In [ ]:
pol_start = train.pol_loss.iloc[0]
pol_end   = train.pol_loss.iloc[-1]
val_start = train.val_loss.iloc[0]
val_end   = train.val_loss.iloc[-1]
mov_start = train.avg_moves.iloc[0]
mov_end   = train.avg_moves.iloc[-1]
total_sp  = train.sp_time_s.sum() / 3600
total_t   = train.iter_time_s.sum() / 3600

print('=== Training Summary ===')
print(f'Iterations           : {len(train)}')
print(f'Policy loss          : {pol_start:.3f} → {pol_end:.3f}  ({(1-pol_end/pol_start)*100:.1f}% reduction)')
print(f'Value loss           : {val_start:.4f} → {val_end:.4f}  ({(1-val_end/val_start)*100:.1f}% reduction)')
print(f'Avg game length      : {mov_start:.0f} → {mov_end:.0f} moves  ({(1-mov_end/mov_start)*100:.1f}% shorter)')
print(f'Total self-play time : {total_sp:.1f} hours')
print(f'Total training time  : {total_t:.1f} hours')
print(f'Total examples       : {train.examples.sum():,}')
print()
print('=== Evaluation Summary ===')
print(f'Eval points          : {len(elo)}')
print(f'Total wins           : {elo.wins.sum()}')
print(f'Total draws          : {elo.draws.sum()}')
print(f'Total losses         : {elo.losses.sum()}')
print(f'Promotions           : {elo.promoted.sum()}')
print(f'Win rate range       : {elo.win_rate.min():.3f} – {elo.win_rate.max():.3f}')

## Figure 1 — Loss Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(7, 2.8), constrained_layout=True)

ax1.plot(train.iteration, train.pol_loss, color='#2c7bb6', linewidth=1.2)
ax1.set_xlabel('Iteration')
ax1.set_ylabel('Cross-entropy loss')
ax1.set_title('Policy Loss')
ax1.set_xlim(1, train.iteration.max())

ax2.plot(train.iteration, train.val_loss, color='#d7191c', linewidth=1.2)
ax2.set_xlabel('Iteration')
ax2.set_ylabel('MSE loss')
ax2.set_title('Value Loss')
ax2.set_xlim(1, train.iteration.max())

fig.savefig(FIGDIR / 'loss_curves.pdf', bbox_inches='tight')
plt.show()
print('Saved: figures/loss_curves.pdf')

## Figure 2 — Game Length Over Training

In [ ]:
fig, ax = plt.subplots(figsize=(5, 2.8), constrained_layout=True)

ax.plot(train.iteration, train.avg_moves, color='#1a9641', linewidth=1.2)
ax.axhline(train.avg_moves.iloc[-1], color='#1a9641', linestyle='--',
           linewidth=0.8, alpha=0.6, label=f'Final: {train.avg_moves.iloc[-1]:.0f} moves')
ax.axhline(train.avg_moves.iloc[0], color='gray', linestyle='--',
           linewidth=0.8, alpha=0.6, label=f'Start: {train.avg_moves.iloc[0]:.0f} moves')
ax.set_xlabel('Iteration')
ax.set_ylabel('Average moves per game')
ax.set_title('Self-Play Game Length')
ax.set_xlim(1, train.iteration.max())
ax.legend(framealpha=0.5)

fig.savefig(FIGDIR / 'game_length.pdf', bbox_inches='tight')
plt.show()
print('Saved: figures/game_length.pdf')

## Figure 3 — Self-Play Throughput

In [ ]:
# Approximate iter/s from self-play time and avg_moves
# outer_iters ≈ avg_moves * num_sims * some_factor (rough)
# Better: just plot sp_time_s as a proxy for efficiency

fig, ax = plt.subplots(figsize=(5, 2.8), constrained_layout=True)

ax.plot(train.iteration, train.sp_time_s, color='#756bb1', linewidth=1.0, alpha=0.6, label='Per iteration')
window = train.sp_time_s.rolling(10, center=True).mean()
ax.plot(train.iteration, window, color='#756bb1', linewidth=2.0, label='10-iter rolling mean')

ax.set_xlabel('Iteration')
ax.set_ylabel('Self-play time (s)')
ax.set_title('Self-Play Wall Time per Iteration')
ax.set_xlim(1, train.iteration.max())
ax.legend(framealpha=0.5)

fig.savefig(FIGDIR / 'sp_time.pdf', bbox_inches='tight')
plt.show()
print('Saved: figures/sp_time.pdf')

## Figure 4 — Evaluation Results

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(7, 2.8), constrained_layout=True)

# Stacked bar: wins / draws / losses per eval
total = elo.wins + elo.draws + elo.losses
ax1.bar(elo.iteration, elo.wins/total,   color='#1a9641', label='Win')
ax1.bar(elo.iteration, elo.draws/total,  color='#bdbdbd', bottom=elo.wins/total, label='Draw')
ax1.bar(elo.iteration, elo.losses/total, color='#d7191c',
        bottom=(elo.wins+elo.draws)/total, label='Loss')
ax1.axhline(0.55, color='black', linestyle='--', linewidth=0.8, label='Promotion threshold')
ax1.set_xlabel('Iteration')
ax1.set_ylabel('Fraction of games')
ax1.set_title('Eval Outcomes (new vs best)')
ax1.legend(fontsize=8, framealpha=0.5)
ax1.set_xlim(elo.iteration.min()-2, elo.iteration.max()+2)

# Win rate over time
ax2.plot(elo.iteration, elo.win_rate, 'o-', color='#2c7bb6',
         markersize=3, linewidth=1.0)
ax2.axhline(0.55, color='black', linestyle='--', linewidth=0.8, label='Promotion (0.55)')
ax2.axhline(0.50, color='gray',  linestyle=':',  linewidth=0.8, label='Even (0.50)')
ax2.set_xlabel('Iteration')
ax2.set_ylabel('Win rate')
ax2.set_title('Eval Win Rate Over Training')
ax2.set_ylim(0.40, 0.65)
ax2.legend(fontsize=8, framealpha=0.5)

fig.savefig(FIGDIR / 'eval_results.pdf', bbox_inches='tight')
plt.show()
print('Saved: figures/eval_results.pdf')

## Figure 5 — Combined Overview (for report cover figure)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(7, 5), constrained_layout=True)

# Policy loss
axes[0,0].plot(train.iteration, train.pol_loss, color='#2c7bb6', linewidth=1.2)
axes[0,0].set_title('(a) Policy Loss')
axes[0,0].set_xlabel('Iteration'); axes[0,0].set_ylabel('CE Loss')

# Value loss
axes[0,1].plot(train.iteration, train.val_loss, color='#d7191c', linewidth=1.2)
axes[0,1].set_title('(b) Value Loss')
axes[0,1].set_xlabel('Iteration'); axes[0,1].set_ylabel('MSE Loss')

# Game length
axes[1,0].plot(train.iteration, train.avg_moves, color='#1a9641', linewidth=1.2)
axes[1,0].set_title('(c) Avg Game Length')
axes[1,0].set_xlabel('Iteration'); axes[1,0].set_ylabel('Moves')

# Eval win rate
axes[1,1].plot(elo.iteration, elo.win_rate, 'o-', color='#756bb1',
               markersize=3, linewidth=1.0)
axes[1,1].axhline(0.55, color='black', linestyle='--', linewidth=0.8)
axes[1,1].axhline(0.50, color='gray',  linestyle=':',  linewidth=0.8)
axes[1,1].set_title('(d) Eval Win Rate')
axes[1,1].set_xlabel('Iteration'); axes[1,1].set_ylabel('Win Rate')
axes[1,1].set_ylim(0.40, 0.65)

for ax in axes.flat:
    ax.set_xlim(1, max(train.iteration.max(), elo.iteration.max()))

fig.savefig(FIGDIR / 'overview.pdf', bbox_inches='tight')
plt.show()
print('Saved: figures/overview.pdf')

## Figure 6 — Loss Rate of Decline (for discussion section)

In [ ]:
# Compute rolling rate of policy loss decline
train['pol_delta'] = -train['pol_loss'].diff()
train['pol_rate']  = train['pol_delta'].rolling(10, center=True).mean()

fig, ax = plt.subplots(figsize=(5, 2.8), constrained_layout=True)
ax.plot(train.iteration, train.pol_rate, color='#2c7bb6', linewidth=1.2)
ax.axhline(0, color='black', linewidth=0.5)
ax.fill_between(train.iteration, train.pol_rate, 0,
                where=train.pol_rate > 0, alpha=0.15, color='#2c7bb6')
ax.set_xlabel('Iteration')
ax.set_ylabel('Δ Policy loss / iteration')
ax.set_title('Rate of Policy Loss Improvement (10-iter rolling mean)')
ax.set_xlim(1, train.iteration.max())

# Annotate the slowdown
ax.axvline(150, color='red', linestyle='--', linewidth=0.8, alpha=0.6)
ax.text(152, ax.get_ylim()[1]*0.85, 'Buffer full\n(iter 150)',
        fontsize=7, color='red', alpha=0.8)

fig.savefig(FIGDIR / 'loss_rate.pdf', bbox_inches='tight')
plt.show()
print('Saved: figures/loss_rate.pdf')